In [ ]:
import calendar
import pandas as pd
import numpy as np

MONTH_NAMES_RU = {
    1: 'Январь', 2: 'Февраль', 3: 'Март', 4: 'Апрель',
    5: 'Май', 6: 'Июнь', 7: 'Июль', 8: 'Август',
    9: 'Сентябрь', 10: 'Октябрь', 11: 'Ноябрь', 12: 'Декабрь'
}

ACTIVITY_MONTHLY_MULTIPLIER = {
    'Daily':  20,  # 5 рабочих дней × 4 недели
    'Weekly':  4,  # 1 раз × 4 недели
}


def normalize_effect(effect, target, current_row):
    if 'mau' in target:
        base = current_row.get('mau', np.nan)
        if pd.isna(base) or base == 0:
            return abs(effect)
        return abs(effect / base * 100)
    else:
        base_col   = target.replace('_next', '')
        base       = current_row.get(base_col, np.nan)
        effect_pct = to_percent_value(effect)
        if pd.isna(base) or base == 0:
            return abs(effect_pct)
        base_pct = to_percent_value(base)
        if base_pct == 0:
            return abs(effect_pct)
        return abs(effect_pct / base_pct * 100)


def build_activity_queue(recommendations, community_type, stage, current_row):
    seen_ids = set()
    queue    = []

    def add_activity(act_id, rationale, priority, predictor, predictor_label, source):
        if act_id in seen_ids or act_id not in df_activities.index:
            return
        act_row = df_activities.loc[act_id]
        freq    = act_row['frequency']
        if freq == 'Once':
            return
        seen_ids.add(act_id)
        queue.append({
            'id':                 act_id,
            'name':               act_row['name'],
            'activity_type':      act_row['activity_type'],
            'work_type':          act_row['work_type'],
            'frequency':          freq,
            'hours_per_activity': act_row['hours_per_activity'],
            'targets':            act_row['targets'],
            'rationale':          rationale,
            'priority':           priority,
            'predictor':          predictor,
            'predictor_label':    predictor_label,
            'source':             source,
        })

    # 1 — рекомендованные
    for rec in recommendations:
        norm = normalize_effect(rec['effect'], rec['target'], current_row)
        acts = get_activities_for_predictor_and_stage(
            rec['predictor'], community_type, stage, df_activities, top_k=10
        )
        for act in acts:
            add_activity(
                act_id          = act['id'],
                rationale       = act['rationale'],
                priority        = norm,
                predictor       = rec['predictor'],
                predictor_label = predictor_labels.get(rec['predictor'], rec['predictor']),
                source          = 'recommended',
            )

    # 2 — остальные из df_predictor_activity
    mask = df_predictor_activity['community_type'].isin([community_type, 'both'])
    for _, row in df_predictor_activity[mask].iterrows():
        add_activity(
            act_id          = row['activity_id'],
            rationale       = row['rationale'],
            priority        = 0,
            predictor       = row['predictor'],
            predictor_label = predictor_labels.get(row['predictor'], row['predictor']),
            source          = 'predictor_linked',
        )

    # 3 — всё остальное из df_activities
    for act_id in df_activities.index:
        add_activity(
            act_id          = act_id,
            rationale       = '',
            priority        = -1,
            predictor       = None,
            predictor_label = None,
            source          = 'filler',
        )

    queue.sort(key=lambda x: x['priority'], reverse=True)
    return queue


def fill_plan(queue, total_hours):
    plan      = []
    remaining = total_hours

    for act in queue:
        multiplier   = ACTIVITY_MONTHLY_MULTIPLIER.get(act['frequency'], 0)
        monthly_cost = act['hours_per_activity'] * multiplier
        if monthly_cost <= 0 or monthly_cost > remaining:
            continue
        plan.append({**act, 'monthly_hours': monthly_cost})
        remaining -= monthly_cost

    return plan, total_hours - remaining


def assign_to_weeks(plan):
    weeks = {1: [], 2: [], 3: [], 4: []}
    for act in plan:
        for w in [1, 2, 3, 4]:
            weeks[w].append(act)
    return weeks


def render_calendar(
    community_id, next_month, community_type, stage,
    recommendations, weeks, total_hours, used_hours, current_row
):
    next_period = pd.Period(next_month, 'M')
    month_name  = MONTH_NAMES_RU[next_period.month]
    year        = next_period.year
    num_days    = calendar.monthrange(year, next_period.month)[1]

    week_ranges = [(1, 7), (8, 14), (15, 21), (22, num_days)]

    lines = [
        '═' * 65,
        f'  ПЛАН АКТИВНОСТЕЙ: {month_name.upper()} {year}',
        f'  Сообщество: {community_id}  │  Тип: {community_type}  │  Стадия: {stage}',
        f'  Бюджет: {used_hours:.1f} ч из {total_hours} ч использовано',
        '═' * 65, '',
        'ФОКУС МЕСЯЦА:',
    ]

    for i, rec in enumerate(recommendations, 1):
        norm    = normalize_effect(rec['effect'], rec['target'], current_row)
        arrow   = '↓' if rec['direction'] == 'decrease' else '↑'
        label   = predictor_labels.get(rec['predictor'], rec['predictor'])
        t_label = target_labels.get(rec['target'], rec['target'])
        lines.append(f'  {i}. {arrow} «{label}»  →  +{norm:.1f}% к {t_label}')
    lines.append('')

    for w_num, (d_start, d_end) in enumerate(week_ranges, 1):
        lines.append(f'НЕДЕЛЯ {w_num}  ({d_start}–{d_end} {month_name[:3].lower()}):')
        week_acts  = weeks[w_num]
        week_hours = 0

        if not week_acts:
            lines.append('  — нет активностей')
        else:
            for act in week_acts:
                h_week = (
                    act['hours_per_activity'] * 5
                    if act['frequency'] == 'Daily'
                    else act['hours_per_activity']
                )
                week_hours += h_week
                freq_tag = 'ежедневно' if act['frequency'] == 'Daily' else 'еженедельно'
                symbol = {'recommended': '★', 'predictor_linked': '◆', 'filler': '·'}.get(act['source'], ' ')
                lines.append(f"  {symbol}[{act['id']}] {act['name']}  [{freq_tag}]  {h_week:.1f}ч")
                if act['predictor_label']:
                    lines.append(f"       → {act['predictor_label']}")
            lines.append(f'  Итого за неделю: {week_hours:.1f}ч')
        lines.append('')

    lines += [
        '─' * 65,
        '★ — рекомендованные  ◆ — связанные с предикторами  · — дополнительные',
        'Важно: baseline без валидации.',
    ]
    return '\n'.join(lines)


def plan_for_community_month(community_id, month, total_hours, top_n=3, return_raw=False):
    community_type, row = detect_community_type(community_id, month)
    stage       = row['stage']
    pred_list   = predictors_chat if community_type == 'chat' else predictors_qa
    current_row = row.to_dict()

    current_values = {p: row[p] for p in pred_list if p in row.index}

    recommendations, _ = build_recommendations(
        community_id   = community_id,
        community_type = community_type,
        stage          = stage,
        current_values = current_values,
        top_n          = top_n,
    )

    queue      = build_activity_queue(recommendations, community_type, stage, current_row)
    plan, used = fill_plan(queue, total_hours)
    weeks      = assign_to_weeks(plan)
    next_month = str(pd.Period(month, 'M') + 1)

    print(render_calendar(
        community_id   = community_id,
        next_month     = next_month,
        community_type = community_type,
        stage          = stage,
        recommendations= recommendations,
        weeks          = weeks,
        total_hours    = total_hours,
        used_hours     = used,
        current_row    = current_row,
    ))

    if return_raw:
        return {
            'community_id':    community_id,
            'next_month':      next_month,
            'community_type':  community_type,
            'stage':           stage,
            'recommendations': recommendations,
            'plan':            plan,
            'weeks':           weeks,
            'total_hours':     total_hours,
            'used_hours':      used,
        }